# PropCare AI — Stage 1: Single-Agent Support Desk with LangChain

Stage 1 treats a tenant request as a focused support-desk problem. One LangChain agent selects from trusted PropCare tools, uses privacy middleware, and returns a Pydantic-validated resolution. This is a clear fit when a request does not need explicit specialist routing or multi-agent orchestration.

## Learning goals

- Inspect the existing `create_agent()` implementation.
- Run two real, read-only PropCare tools against fictional tenant data.
- See the built-in `PIIMiddleware` and `TenantResolution` schema used by production code.
- Follow the flow: **Tenant Request → Single Agent → Tool Selection → Property Data → Structured Resolution**.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'backend').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TENANT_ID = 'T-1001'
TENANT_MESSAGE = 'Can you check whether my rent for this month has been paid?'
print(f'Project root: {ROOT}')
print(f'Tenant message: {TENANT_MESSAGE}')

In [ ]:
from backend.tools.propcare_tools import PROPCare_TOOLS, check_rent_status, lookup_tenant

# These are the same LangChain tools registered with the Stage 1 agent.
tenant = lookup_tenant.invoke({'tenant_id': TENANT_ID})
payment = check_rent_status.invoke({'tenant_id': TENANT_ID})

print('Registered tools:', [tool.name for tool in PROPCare_TOOLS])
print('\nTenant lookup:', tenant)
print('\nRent-status lookup:', payment)

## The production agent

`backend.agent.propcare_agent.build_agent()` uses LangChain `create_agent()` with the PropCare tool collection. It attaches the built-in `PIIMiddleware` twice: email and phone-number redaction apply to both input and output. The model is configured with `temperature=0`, an explicit timeout, and `max_retries=0` so a retry cannot accidentally repeat a tool action.

In [ ]:
from backend.agent.propcare_agent import PIIMiddleware, build_agent
from backend.config import load_environment

load_environment()
print('Middleware class:', PIIMiddleware.__name__)
if os.getenv('OPENAI_API_KEY'):
    stage1_agent = build_agent()
    print('Built agent type:', type(stage1_agent).__name__)
else:
    stage1_agent = None
    print('No OPENAI_API_KEY found: the real tools above still ran; add a key to build the live model-backed agent.')

In [ ]:
from backend.schemas.models import TenantResolution

# This is a schema preview, not a model-generated response. Production validates
# the agent's `structured_response` against this same Pydantic model.
schema_preview = TenantResolution.model_validate({
    'tenant_id': TENANT_ID,
    'request_id': None,
    'issue_category': 'billing',
    'priority': 'low',
    'assigned_team': 'Billing',
    'action_taken': 'Checked the latest rent/payment record.',
    'approval_required': False,
    'status': 'resolved',
    'summary': f"Payment status: {payment.get('payment_status', 'unknown')}.",
})
print(schema_preview.model_dump())

## Optional live model call

Set `RUN_LIVE_MODEL = True` only after configuring `OPENAI_API_KEY`. The call below sends the example tenant message through the real Stage 1 agent and prints the validated `TenantResolution`. It is left opt-in because a live model can select a mutating tool for other prompts.

In [ ]:
from backend.agent.propcare_agent import resolve_tenant_message

RUN_LIVE_MODEL = False
if RUN_LIVE_MODEL:
    live_resolution = resolve_tenant_message(TENANT_ID, TENANT_MESSAGE)
    print(live_resolution.model_dump())
else:
    print('Live call skipped. Set RUN_LIVE_MODEL = True to invoke LangChain with your configured provider key.')

## Stage 1 boundary

Stage 1 deliberately has **no specialist routing, graph state, supervisor loop, or multi-agent orchestration**. Those controls are introduced in Stage 2 when a request needs several domain reviews and a human approval pause.